In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ellipticco/elliptic-data-set")

print("Path to dataset files:", path)

100%|██████████| 146M/146M [00:09<00:00, 16.1MB/s] 

Extracting files...


Path to dataset files: C:\Users\yucha\.cache\kagglehub\datasets\ellipticco\elliptic-data-set\versions\1


In [20]:
path = r"C:\Users\yucha\.cache\kagglehub\datasets\ellipticco\elliptic-data-set\versions\1\elliptic_bitcoin_dataset"

import pandas as pd
import os

files = []
for file in os.listdir(path):
    files.append(os.path.join(path, file))
    print(files)

df1 = pd.read_csv(files[0])
df2 = pd.read_csv(files[1])
df3 = pd.read_csv(files[2], header=None)
df3.columns = (
    ['txId', 'time_step'] +
    [f'feat_{i}' for i in range(df3.shape[1] - 2)]
)

['C:\\Users\\yucha\\.cache\\kagglehub\\datasets\\ellipticco\\elliptic-data-set\\versions\\1\\elliptic_bitcoin_dataset\\elliptic_txs_classes.csv']
['C:\\Users\\yucha\\.cache\\kagglehub\\datasets\\ellipticco\\elliptic-data-set\\versions\\1\\elliptic_bitcoin_dataset\\elliptic_txs_classes.csv', 'C:\\Users\\yucha\\.cache\\kagglehub\\datasets\\ellipticco\\elliptic-data-set\\versions\\1\\elliptic_bitcoin_dataset\\elliptic_txs_edgelist.csv']
['C:\\Users\\yucha\\.cache\\kagglehub\\datasets\\ellipticco\\elliptic-data-set\\versions\\1\\elliptic_bitcoin_dataset\\elliptic_txs_classes.csv', 'C:\\Users\\yucha\\.cache\\kagglehub\\datasets\\ellipticco\\elliptic-data-set\\versions\\1\\elliptic_bitcoin_dataset\\elliptic_txs_edgelist.csv', 'C:\\Users\\yucha\\.cache\\kagglehub\\datasets\\ellipticco\\elliptic-data-set\\versions\\1\\elliptic_bitcoin_dataset\\elliptic_txs_features.csv']


In [22]:
print("DF 1 Information")
print(f"Columns: {df1.columns}")
print(f"Dtypes: {df1.dtypes}")
print()

print("DF 2 Information")
print(f"Columns: {df2.columns}")
print(f"Dtypes: {df2.dtypes}")
print()

print("DF 3 Information")
print(f"Columns: {df3.columns}")
print(f"Dtypes: {df3.dtypes}")
print()

DF 1 Information
Columns: Index(['txId', 'class'], dtype='str')
Dtypes: txId     int64
class      str
dtype: object

DF 2 Information
Columns: Index(['txId1', 'txId2'], dtype='str')
Dtypes: txId1    int64
txId2    int64
dtype: object

DF 3 Information
Columns: Index(['txId', 'time_step', 'feat_0', 'feat_1', 'feat_2', 'feat_3', 'feat_4',
       'feat_5', 'feat_6', 'feat_7',
       ...
       'feat_155', 'feat_156', 'feat_157', 'feat_158', 'feat_159', 'feat_160',
       'feat_161', 'feat_162', 'feat_163', 'feat_164'],
      dtype='str', length=167)
Dtypes: txId           int64
time_step      int64
feat_0       float64
feat_1       float64
feat_2       float64
              ...   
feat_160     float64
feat_161     float64
feat_162     float64
feat_163     float64
feat_164     float64
Length: 167, dtype: object



In [24]:
# Merging df1 and df3 into df4
df4 = df3.merge(df1, on='txId')

print(df4.head(1))

        txId  time_step    feat_0    feat_1    feat_2   feat_3    feat_4  \
0  230425980          1 -0.171469 -0.184668 -1.201369 -0.12197 -0.043875   

     feat_5    feat_6    feat_7  ...  feat_156  feat_157  feat_158  feat_159  \
0 -0.113002 -0.061584 -0.162097  ... -0.600999   1.46133  1.461369  0.018279   

   feat_160  feat_161  feat_162  feat_163  feat_164    class  
0  -0.08749 -0.131155 -0.097524 -0.120613 -0.119792  unknown  

[1 rows x 168 columns]


In [27]:
%pip install torch

  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   - -------------------------------------- 3.9/114.6 MB 21.5 MB/s eta 0:00:06
   --- ------------------------------------ 10.7/114.6 MB 26.3 MB/s eta 0:00:04
   ----- ---------------------------------- 16.5/114.6 MB 26.9 MB/s eta 0:00:04
   ------- -------------------------------- 22.3/114.6 MB 26.9 MB/s eta 0:00:04
   --------- ------------------------------ 28.3/114.6 MB 27.3 MB/s eta 0:00:04
   ------------ --------------------------- 34.6/114.6 MB 27.3 MB/s eta 0:00:03
   -------------- ------------------------- 40.6/114.6 MB 27.7 MB/s eta 0:00:03
   ---------------- ----------------------- 47.2/114.6 MB 28.1 MB/s eta 0:00:03
   ------------------- -----------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
# Implementing graph edges

# Start by determining all valid nodes featured in df4
valid_nodes = set(df4['txId'])

# Check if nodes in valid_nodes set
df2 = df2[
    df2['txId1'].isin(valid_nodes) &
    df2['txId2'].isin(valid_nodes)
]

# Node -> Index mapping
node_to_idx = {node: i for i, node in enumerate(df4['txId'])}

# Convert edges to indices form
edges = []

for _, row in df2.iterrows():
    i = node_to_idx[row['txId1']]
    j = node_to_idx[row['txId2']]
    edges.append((i, j))
    edges.append((j, i))  # make undirected

# Edge Index
import torch

edge_index = torch.tensor(edges, dtype=torch.long).t()

In [31]:
# Edge indexing shape check
print(edge_index.shape)

# Node count vs max index
num_nodes = len(df4)
print(edge_index.max(), num_nodes)

# Check edges
print(len(edges))

torch.Size([2, 468710])
tensor(203768) 203769
468710


In [ ]:
# Prepare data for classification model
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Merge df1 (labels) and df3 (features)
# df3 has features in columns, so we use it directly as X
X = df3.copy()
y = df1['class'].copy()

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Label distribution:\n{y.value_counts()}")

# Check for missing values
print(f"\nMissing values in features: {X.isnull().sum().sum()}")
print(f"Missing values in labels: {y.isnull().sum()}")

# Encode labels if necessary
if y.dtype == 'object':
    le = LabelEncoder()
    y = le.fit_transform(y)
    print(f"\nEncoded classes: {le.classes_}")

In [16]:
%pip install scikit-learn
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Train-test split and model training
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Training label distribution:\n{np.bincount(y_train)}")

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=15)
rf_model.fit(X_train, y_train)

# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train, y_train)

print("Models trained successfully!")

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   --------- ------------------------------ 1.8/8.1 MB 30.2 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 28.9 MB/s  0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


NameError: name 'X' is not defined

In [ ]:
# Model evaluation
rf_pred = rf_model.predict(X_test)
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]

lr_pred = lr_model.predict(X_test)
lr_pred_proba = lr_model.predict_proba(X_test)[:, 1]

# Random Forest results
print("=" * 50)
print("RANDOM FOREST RESULTS")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, rf_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, rf_pred_proba):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

# Logistic Regression results
print("\n" + "=" * 50)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, lr_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, lr_pred_proba):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_pred))

In [ ]:
# Visualization of results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Random Forest Confusion Matrix
cm_rf = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0])
axes[0, 0].set_title('Random Forest - Confusion Matrix')
axes[0, 0].set_ylabel('True Label')
axes[0, 0].set_xlabel('Predicted Label')

# Logistic Regression Confusion Matrix
cm_lr = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens', ax=axes[0, 1])
axes[0, 1].set_title('Logistic Regression - Confusion Matrix')
axes[0, 1].set_ylabel('True Label')
axes[0, 1].set_xlabel('Predicted Label')

# ROC Curves
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_pred_proba)
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_pred_proba)

axes[1, 0].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={roc_auc_score(y_test, rf_pred_proba):.3f})', linewidth=2)
axes[1, 0].plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={roc_auc_score(y_test, lr_pred_proba):.3f})', linewidth=2)
axes[1, 0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1, 0].set_xlabel('False Positive Rate')
axes[1, 0].set_ylabel('True Positive Rate')
axes[1, 0].set_title('ROC Curves')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Feature Importance (Random Forest)
feature_importance = rf_model.feature_importances_
top_n = 20
top_features_idx = np.argsort(feature_importance)[-top_n:]
top_features_importance = feature_importance[top_features_idx]
top_features_names = [str(X.columns[i]) for i in top_features_idx]

axes[1, 1].barh(range(top_n), top_features_importance)
axes[1, 1].set_yticks(range(top_n))
axes[1, 1].set_yticklabels(top_features_names, fontsize=8)
axes[1, 1].set_xlabel('Importance')
axes[1, 1].set_title('Top 20 Feature Importance (Random Forest)')
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()